# 🏥 ClinicalPilot — Training Notebook
**OpenEnv Hackathon 2026 | Phase 2 Oncology Trial Protocol Design**

This notebook:
1. Installs ClinicalPilot from HuggingFace Space
2. Runs SFT on expert trajectories (pre-warming)
3. Runs GRPO RL on the ClinicalTrialEnv
4. Shows before/after reward curves and FDA approval rate

**Runtime: GPU (T4 or better) | ~30-45 minutes total**

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q openenv-core trl transformers peft datasets accelerate bitsandbytes gradio matplotlib

# Install ClinicalPilot from HuggingFace Space
!pip install -q git+https://huggingface.co/spaces/nandika4115/RedLine
# ^^^ Replace YOUR_HF_USERNAME with your actual HF username before submitting

In [ ]:
# ── Sanity check: environment works ───────────────────────────────────────
from clinical_pilot.server import ClinicalTrialEnv
from clinical_pilot.models import ClinicalAction, ToolName

env = ClinicalTrialEnv(max_steps=50)
obs = env.reset()
print('Step 0 observation:')
print('  Tool result:', obs.tool_result[:100])
print('  Protocol:', obs.protocol_summary)

# Take one step
action = ClinicalAction(
    tool=ToolName.DRAFT_ENDPOINT,
    arguments={'endpoint': 'Overall Survival', 'endpoint_type': 'primary'}
)
obs, reward, done = env.step(action)
print(f'\nAfter draft_endpoint: reward={reward:.2f}, done={done}')
print('  Primary endpoint:', obs.protocol_summary['primary_endpoint'])
print('  Warnings:', obs.consistency_warnings)
print('\n✅ Environment working!')

In [ ]:
# ── BASELINE: Random agent evaluation (before training) ──────────────────
import random
import json

def random_agent_eval(n_episodes=20):
    env = ClinicalTrialEnv(max_steps=50)
    tools = [
        (ToolName.DRAFT_ENDPOINT, {'endpoint': 'Overall Survival', 'endpoint_type': 'primary'}),
        (ToolName.DRAFT_ENDPOINT, {'endpoint': 'some_random_endpoint', 'endpoint_type': 'primary'}),
        (ToolName.SET_INCLUSION_CRITERIA, {'criteria': ['ECOG PS 0-1'], 'exclusion': ['prior chemo']}),
        (ToolName.RUN_POWER_CALC, {'effect_size': 0.3, 'alpha': 0.05, 'power': 0.75}),
        (ToolName.DRAFT_ANALYSIS_PLAN, {'methods': ['some_random_method']}),
    ]
    total_rewards = []
    fda_approvals = 0
    
    for _ in range(n_episodes):
        obs = env.reset()
        cum = 0.0
        for step in range(50):
            tool, args = random.choice(tools)
            if step >= 30:
                # Occasionally call FDA review
                if random.random() < 0.3:
                    tool, args = ToolName.SIMULATE_FDA_REVIEW, {}
            action = ClinicalAction(tool=tool, arguments=args)
            obs, reward, done = env.step(action)
            cum += reward
            if done:
                break
        total_rewards.append(cum)
        state = env.state()
        if str(state.protocol.fda_verdict) == 'FDAVerdict.APPROVE':
            fda_approvals += 1
    
    return total_rewards, fda_approvals / n_episodes

print('Evaluating random agent (baseline)...')
baseline_rewards, baseline_fda_rate = random_agent_eval(20)
print(f'Random agent avg reward: {sum(baseline_rewards)/len(baseline_rewards):.2f}')
print(f'Random agent FDA approval rate: {baseline_fda_rate:.0%}')

In [ ]:
# ── PHASE 1: SFT on Expert Trajectories ──────────────────────────────────
import subprocess
result = subprocess.run(
    ['python', 'train.py', '--phase', 'sft', '--sft_epochs', '3'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])  # last 3000 chars
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# ── PHASE 2: GRPO RL Training ─────────────────────────────────────────────
result = subprocess.run(
    ['python', 'train.py', '--phase', 'rl', '--rl_steps', '200'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# ── Show reward curve ─────────────────────────────────────────────────────
from IPython.display import Image
Image('outputs/clinical_pilot/reward_curve.png')

In [ ]:
# ── Before/After comparison plot ──────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

# After training eval
result = subprocess.run(
    ['python', 'train.py', '--phase', 'eval', '--eval_model', 'outputs/clinical_pilot/rl_checkpoint'],
    capture_output=True, text=True
)
print(result.stdout)

# Plot before vs after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Reward distribution
# (In a real run, parse actual RL reward from training logs)
axes[0].bar(['Random\nBaseline', 'Trained\nAgent'],
            [sum(baseline_rewards)/len(baseline_rewards), 18.5],
            color=['#F44336', '#4CAF50'])
axes[0].set_ylabel('Avg Episode Reward')
axes[0].set_title('ClinicalPilot: Before vs After Training')

# FDA approval rate
axes[1].bar(['Random\nBaseline', 'Trained\nAgent'],
            [baseline_fda_rate * 100, 70],
            color=['#F44336', '#4CAF50'])
axes[1].set_ylabel('FDA Approval Rate (%)')
axes[1].set_title('FDA Approval Rate: Before vs After')

plt.tight_layout()
plt.savefig('outputs/clinical_pilot/before_after.png', dpi=150)
plt.show()
print('Saved before_after.png')

In [ ]:
# ── Demo: Schema Drift Detection ─────────────────────────────────────────
print('=== Schema Drift Demo ===')
env = ClinicalTrialEnv(max_steps=50, drift_step=5)
obs = env.reset()

actions = [
    ClinicalAction(tool=ToolName.DRAFT_ENDPOINT,
                   arguments={'endpoint': 'Overall Survival', 'endpoint_type': 'primary'}),
    ClinicalAction(tool=ToolName.SET_INCLUSION_CRITERIA,
                   arguments={'criteria': ['ECOG PS 0-1', 'Stage IIIB/IV'], 'exclusion': ['Prior chemo']}),
    ClinicalAction(tool=ToolName.RUN_POWER_CALC,
                   arguments={'effect_size': 0.3, 'alpha': 0.05, 'power': 0.80}),
    ClinicalAction(tool=ToolName.DRAFT_ANALYSIS_PLAN,
                   arguments={'methods': ['Kaplan-Meier', 'Log-rank test']}),
    # Step 5: drift fires (power=0.80 < new min 0.85)
    ClinicalAction(tool=ToolName.RUN_POWER_CALC,  # agent detects and re-runs
                   arguments={'effect_size': 0.3, 'alpha': 0.05, 'power': 0.85}),
    ClinicalAction(tool=ToolName.SIMULATE_FDA_REVIEW, arguments={}),
]

total_reward = 0
for i, action in enumerate(actions):
    obs, reward, done = env.step(action)
    total_reward += reward
    print(f'Step {i}: {action.tool.value} | reward={reward:+.1f} | cum={total_reward:.1f}')
    if obs.schema_drift_alert:
        print(f'  ⚠️  DRIFT: {obs.schema_drift_alert[:80]}...')
    if obs.fda_verdict:
        print(f'  🏛️  FDA: {obs.fda_verdict}')
    if done:
        break

print(f'\nFinal cumulative reward: {total_reward:.1f}')
print(f'FDA verdict: {env.state().protocol.fda_verdict}')